In [1]:
import os
import pandas as pd
import subprocess

# Load the CSV file with Subject ID and T1 Series ID
subject_csv = "E:/Datasets/Re-Creation/Selected Data/selected_data.csv"
subject_metadata = pd.read_csv(subject_csv)

# Load the SEG metadata CSV file
seg_csv = "E:/Datasets/updated_file.csv"
seg_metadata = pd.read_csv(seg_csv)

# Base directories
dicom_base_dir = "E:/Datasets/manifest-DICOM-GLIOMA-SEG_2024"  # Base directory for DICOM files
nifti_base_dir = "E:/Datasets/Re-Creation/Selected Data"  # Base directory for NIFTI files


# Function to convert DICOM Segmentation Objects to NIFTI using dcmqi
def convert_seg_to_nifti(seg_dicom_path, nifti_dir, output_name):
    os.makedirs(nifti_dir, exist_ok=True)
    
     # Run dcmqi's itkimage2segimage
    output_path = os.path.join(nifti_dir, f"{output_name}")
    command = f'segimage2itkimage --inputDICOM "{seg_dicom_path}" --outputDirectory "{nifti_dir}" --prefix "{output_name}"  --outputType "nifti"  --mergeSegments '
    print(f"Running command: {command}") 

    subprocess.run(command, shell=True, check=True)
    # Compress the NIFTI file to .nii.gz
    path = f"{output_path}-1.nii.gz"
    return path
    

# Add a new column to the subject metadata for the SEG file path
subject_metadata["SEG File Path"] = None

# Iterate through each SEG file
for _, seg_row in seg_metadata.iterrows():
    seg_series_id = seg_row["series_id"]
    print(f"Processing SEG for Series ID: {seg_series_id}...")

    # Find the corresponding subject folder using the Series ID
    matching_subject = subject_metadata[subject_metadata["T1 Series ID"] == seg_series_id]
    if not matching_subject.empty:
        seg_dicom_dir = os.path.join(dicom_base_dir ,seg_row["File Path"])
        seg_dicom_dir = seg_dicom_dir.replace(".\\","/")
        seg_dicom_dir = seg_dicom_dir.replace("\\","/")
        seg_dicom_dir = seg_dicom_dir.replace("//","/")
        seg_dicom_dir = os.path.join(seg_dicom_dir ,"1-1.dcm")
        subject_id = matching_subject["Subject ID"].iloc[0]
        subject_folder = os.path.join(nifti_base_dir, f"{subject_id}")
        if os.path.exists(subject_folder):
            # Create SEG directory
            seg_nifti_dir = os.path.join(subject_folder, "SEG")
            os.makedirs(seg_nifti_dir, exist_ok=True)
            # Convert SEG DICOM to NIFTI
            seg_nifti_path = convert_seg_to_nifti(seg_dicom_dir, seg_nifti_dir, "seg")
            if seg_nifti_path:
                print(f"SEG file saved to: {seg_nifti_path}")
                # Update the SEG File Path in the subject metadata
                
                subject_metadata.loc[matching_subject.index, "SEG File Path"] = seg_nifti_path
            else:
                print(f"Failed to convert SEG for Series ID: {seg_series_id}")
        else:
            print(f"Subject folder not found for Subject ID: {subject_id}")
    else:
        print(f"no match")

subject_metadata.to_csv("E:/Datasets/Re-Creation/Selected Data/selected_data.csv", index = False)

Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.7695.4001.224301481773938865793646319386...
Running command: segimage2itkimage --inputDICOM "E:/Datasets/manifest-DICOM-GLIOMA-SEG_2024/TCGA-GBM/TCGA-08-0509/11-13-1997-NA-MSTEALTH-79103/300.000000-FMPSPGR SAG Glioma Segmentation-97872\1-1.dcm" --outputDirectory "E:/Datasets/Re-Creation/Selected Data\TCGA-08-0509\SEG" --prefix "seg"  --outputType "nifti"  --mergeSegments 
SEG file saved to: E:/Datasets/Re-Creation/Selected Data\TCGA-08-0509\SEG\seg-1.nii.gz
Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.7695.4001.408596602542998666261259876805...
no match
Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.7695.4001.258664400961230551340675293298...
no match
Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.7695.4001.178602050598278251326371799672...
no match
Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.7695.4001.147430811790810817730832339214...
no match
Processing SEG for Series ID: 1.3.6.1.4.1.14519.5.2.1.76